In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LFM_testbench").config("spark.sql.shuffle.partitions", "4000").config("spark.executor.memoryOverhead", "4g").getOrCreate()

bash: /opt/conda/miniconda3/lib/libtinfo.so.6: no version information available (required by bash)
bash: /opt/conda/miniconda3/lib/libtinfo.so.6: no version information available (required by bash)
26/03/23 05:31:03 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 05:31:03 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/23 05:31:03 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Pleas

In [2]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F

date = "2026-03-21"
train_days = 30
test_days = 10

In [3]:
daily_watch_history_path = "gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new"
date_format = "%Y-%m-%d"
base_date = datetime.strptime(date, date_format)

# 1. Test Paths: The last 30 days (0 to 29 days ago)
test_paths = [
    f"{daily_watch_history_path}/day={(base_date - timedelta(days=i)).strftime(date_format)}"
    for i in range(test_days)
]

# 2. Train Paths: The 90 days before the test period (30 to 119 days ago)
train_paths = [
    f"{daily_watch_history_path}/day={(base_date - timedelta(days=i)).strftime(date_format)}"
    for i in range(test_days, test_days + train_days)
]

test_df = None
train_df = None

for path in test_paths:
    try:
        temp_df = spark.read.parquet(path)

        if test_df is None:
            test_df = temp_df
        else:
            test_df = test_df.unionByName(temp_df)

    except Exception:
        print(f"Skipping missing TEST path: {path}")

for path in train_paths:
    try:
        temp_df = spark.read.parquet(path)

        if train_df is None:
            train_df = temp_df
        else:
            train_df = train_df.unionByName(temp_df)

    except Exception:
        print(f"Skipping missing TRAIN path: {path}")

In [4]:
enriched_db_path = "gs://wynk-ml-workspace/projects/xstream_nlu/catalog-db/" 

enriched_tv_df = spark.read.parquet(f'{enriched_db_path}{date}/enriched_tv.parquet')
enriched_movies_df = spark.read.parquet(f'{enriched_db_path}{date}/enriched_movie.parquet')

filtered_enriched_tv = (
    enriched_tv_df
    # 1. Remove empty arrays and un-published content
    .filter((F.col('XstreamContentIds') != F.array()) & (F.col("published") == True))
    # 2. Explode and select in one go
    .select(
        F.explode("XstreamContentIds").alias("item_id"), # This is your primary key for ALS joins
        "title", 
        "ID",
        F.col('OriginalLanguage').alias('original_language'), 
        "Genres"
    )
)
filtered_enriched_movies = (
    enriched_movies_df
    # 1. Remove empty arrays and un-published content
    .filter((F.col('XstreamContentIds') != F.array()) & (F.col("published") == True))
    # 2. Explode and select in one go
    .select(
        F.explode("XstreamContentIds").alias("item_id"), # This is your primary key for ALS joins
        "title", 
        "ID", 
        F.col('OriginalLanguage').alias('original_language'),
        "Genres"
    )
)

# 1. Ignore files that are physically broken/incomplete
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")

# 2. Disable the Vectorized Reader (Standardizes the reading process)
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")

# 3. Handle schema mismatches if they exist
spark.conf.set("spark.sql.parquet.mergeSchema", "true")

# 4. Perform the union with explicit casting to ensure types match
tv_final = filtered_enriched_tv.select(
    F.col("item_id").cast("string"),
    F.col("title").cast("string"),
    F.col("original_language").cast("string"),
    F.col("Genres").cast("string") # Ensure both are cast to the same type
)

movies_final = filtered_enriched_movies.select(
    F.col("item_id").cast("string"),
    F.col("title").cast("string"),
    F.col("original_language").cast("string"),
    F.col("Genres").cast("string")
)

enriched_content_df = tv_final.unionByName(movies_final).distinct()
# enriched_content_df.cache().count() # Use count() to force Spark to read and verify all files now
enriched_content_df = filtered_enriched_tv.unionByName(filtered_enriched_movies)


26/03/23 05:31:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [5]:
# 2. Get unique users (this is a heavy operation, so we do it once)
# Using .distinct() on 500GB will trigger a shuffle.
train_users = train_df.select("userId").distinct()
test_users = test_df.select("userId").distinct()

# 3. Join them to find the 'overlap'
common_users = train_users.join(test_users, on="userId", how="inner")

# 4. Filter the main dataframes
# We use an inner join here because it's generally more 
# efficient than 'isin' in Spark for large datasets.
df_train_filtered = train_df.join(common_users, on="userId", how="inner")
df_test_filtered = test_df.join(common_users, on="userId", how="inner")

In [6]:
import pandas as pd
import numpy as np
import ast
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from lightfm.data import Dataset
from lightfm import LightFM
from lightfm.evaluation import precision_at_k, recall_at_k

# =====================================================================
# 1. SPARK: AGGREGATE TRAIN DATA
# =====================================================================
print("Aggregating Spark Train Data...")
def compute_user_stats(watch_history_df):
    user_stats = (
        watch_history_df  
        .groupBy("userId", "item_id")
        .agg(F.sum("total_play_time_sec").alias("total_playtime_combined"))
        .withColumn("distinct_content_count", F.count("item_id").over(Window.partitionBy("userId")))
        .filter(F.col("total_playtime_combined").isNotNull() & ~F.isnan("total_playtime_combined"))
    )
    return user_stats

train_combined_stats = compute_user_stats(df_train_filtered)
print("Sample of aggregated train data:")
train_combined_stats.show(5, truncate=False)

# Filter for users with at least 2 distinct items
als_input_base = train_combined_stats.filter("distinct_content_count >= 2").cache()

# =====================================================================
# 2. SPARK: FIND OVERLAPPING USERS & FILTER TEST DATA
# =====================================================================
print("Filtering Test Data for Overlapping Users...")
test_users = df_test_filtered.select("userId").distinct()
re_common_users = test_users.join(als_input_base.select("userId").distinct(), on="userId", how="inner").cache()

df_test_filtered = df_test_filtered.join(re_common_users, on="userId", how="inner")
print("df_test_data_filtered:")
df_test_filtered.show(5, truncate=False)

# =====================================================================
# 3. SPARK: CREATE CUSTOM LOOKUPS & APPLY TO TRAIN/TEST
# =====================================================================
print("Creating RDD Lookups and Indexing Data...")
distinct_users = als_input_base.select("userId").distinct()
distinct_items = als_input_base.select("item_id").distinct()

user_lookup = (
    distinct_users.rdd.zipWithIndex()
    .toDF(["user_struct", "userIndex"])
    .select(F.col("user_struct.*"), F.col("userIndex").cast("int"))
)

item_lookup = (
    distinct_items.rdd.zipWithIndex()
    .toDF(["item_struct", "itemIndex"])
    .select(F.col("item_struct.*"), F.col("itemIndex").cast("int"))
)

# Apply lookups to Train
df_train_indexed = (
    als_input_base
    .join(user_lookup, on="userId", how="inner")
    .join(item_lookup, on="item_id", how="inner")
    .withColumn("playtime_logged", F.log1p("total_playtime_combined"))
)
print("Sample of indexed train data:")
df_train_indexed.show(5, truncate=False)

# Apply lookups to Test
df_test_indexed = (
    df_test_filtered
    .join(user_lookup, on="userId", how="inner")
    .join(item_lookup, on="item_id", how="inner")
    # Assuming test data also has 'total_play_time_sec' to be logged
    .withColumn("playtime_logged", F.log1p("total_play_time_sec")) 
)

# =====================================================================
# 3.5 SPARK: SCRUB TEST DATA OF ALREADY-SEEN ITEMS
# =====================================================================
print("Removing train user-item pairs from the test set...")

# A left_anti join drops any row in df_test_indexed where the exact 
# same userIndex and itemIndex pair already exists in df_train_indexed.
df_test_indexed = df_test_indexed.join(
    df_train_indexed.select("userIndex", "itemIndex"),
    on=["userIndex", "itemIndex"],
    how="left_anti"
)

# =====================================================================
# 4. SPARK TO PANDAS: SAFE SAMPLING FOR LIGHTFM
# =====================================================================
print("Sampling Data for LightFM (to prevent Driver OOM)...")
# WARNING: LightFM runs in memory on the driver. We MUST sample the 500GB data.
# We sample 5% of users. Adjust this fraction based on your driver's RAM.
sampled_users = re_common_users.sample(withReplacement=False, fraction=0.05, seed=42)

sampled_train_spark = df_train_indexed.join(sampled_users, on="userId", how="inner")
sampled_test_spark = df_test_indexed.join(sampled_users, on="userId", how="inner")

print("Fetching interactions from Spark to Pandas...")
train_pdf = sampled_train_spark.select("userIndex", "itemIndex", "playtime_logged").toPandas()
test_pdf = sampled_test_spark.select("userIndex", "itemIndex", "playtime_logged").toPandas()

print("Train_pdf sample:")
train_pdf.head()

print("Fetching items metadata from Spark to Pandas...")
items_pdf = (
    enriched_content_df.join(item_lookup, "item_id")
    .select("itemIndex", "item_id", "title", "original_language", "Genres")
    .dropDuplicates(["itemIndex"])
    .toPandas()
)
print("Items_pdf sample uncleaned:")
items_pdf.head()

# =====================================================================
# 5. PANDAS: CLEAN DATA & FEATURE EXTRACTION
# =====================================================================
print("Cleaning missing values and building features...")
items_pdf['Genres'] = items_pdf['Genres'].fillna('Unknown').astype(str)
items_pdf['original_language'] = items_pdf['original_language'].fillna('Unknown').astype(str)

# =====================================================================
# FILTER ORPHAN ITEMS OUT OF METADATA
# =====================================================================
print("Filtering orphan items from metadata...")

# Find all unique items present in our sampled train and test sets
valid_sampled_items = np.unique(np.concatenate((train_pdf['itemIndex'], test_pdf['itemIndex'])))

# Filter the items_pdf to only include these valid items
items_pdf = items_pdf[items_pdf['itemIndex'].isin(valid_sampled_items)]

print(f" -> Reduced items metadata to {len(items_pdf)} valid items.")
print("Sample of cleaned items metadata:")
items_pdf.head()

def extract_flat_features(lang, genre_data):
    features = [str(lang)]
    if isinstance(genre_data, str):
        if genre_data.startswith('['):
            try:
                genre_data = ast.literal_eval(genre_data)
            except:
                genre_data = genre_data.replace('[', '').replace(']', '').replace("'", "").replace('"', "")
                genre_data = [g.strip() for g in genre_data.split(',')]
        else:
            genre_data = [g.strip() for g in genre_data.split(',')]
    if isinstance(genre_data, list):
        for item in genre_data:
            if isinstance(item, list):
                features.extend([str(i).strip() for i in item])
            else:
                features.append(str(item).strip())
    return features

items_pdf['clean_features'] = items_pdf.apply(
    lambda row: extract_flat_features(row['original_language'], row['Genres']), axis=1
)

all_item_features = list(set([feat for sublist in items_pdf['clean_features'] for feat in sublist]))
print(f"Extracted {len(all_item_features)} unique item features.")
print("Sample of extracted item features:")
print(all_item_features[:10])

# =====================================================================
# 6. LIGHTFM: INITIALIZE DATASET & BUILD MATRICES
# =====================================================================
print("Fitting LightFM Dataset mapping...")
# CRITICAL: Fit the dataset on the combined unique IDs from train and test
all_users = np.unique(np.concatenate((train_pdf['userIndex'], test_pdf['userIndex'])))
all_items = np.unique(np.concatenate((train_pdf['itemIndex'], test_pdf['itemIndex'])))

dataset = Dataset()
dataset.fit(
    users=all_users,
    items=all_items,
    item_features=all_item_features
)

print("Building Interactions matrices...")
(train_interactions, train_weights) = dataset.build_interactions(
    zip(train_pdf['userIndex'], train_pdf['itemIndex'], train_pdf['playtime_logged'])
)

(test_interactions, _) = dataset.build_interactions(
    zip(test_pdf['userIndex'], test_pdf['itemIndex'], test_pdf['playtime_logged'])
)

print("Building Item Features matrix...")
item_features = dataset.build_item_features(
    (item_idx, features) for item_idx, features in zip(items_pdf['itemIndex'], items_pdf['clean_features'])
)

print("cell executed successfully!")



Aggregating Spark Train Data...
Sample of aggregated train data:


+------------------+----------------------------+-----------------------+----------------------+
|userId            |item_id                     |total_playtime_combined|distinct_content_count|
+------------------+----------------------------+-----------------------+----------------------+
|--JRN2i6pSclbcw800|SONYLIV_VOD_MOVIE_1090503578|9078.028999999999      |20                    |
|--JRN2i6pSclbcw800|ZEEFIVE_MOVIE_0-0-1z5906815 |1004.0                 |20                    |
|--JRN2i6pSclbcw800|ZEEFIVE_TVSHOW_0-6-4z5594936|3417.0                 |20                    |
|--JRN2i6pSclbcw800|MANORAMAMAX_MOVIE_10000099  |44.361                 |20                    |
|--JRN2i6pSclbcw800|MANORAMAMAX_MOVIE_10000097  |8825.055               |20                    |
+------------------+----------------------------+-----------------------+----------------------+
only showing top 5 rows

Filtering Test Data for Overlapping Users...
df_test_data_filtered:


26/03/23 05:31:54 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 05:32:15 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 05:32:16 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 05:32:34 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


+------------------+---------------------------------------------------------------+-------------------+----------+
|userId            |item_id                                                        |total_play_time_sec|day       |
+------------------+---------------------------------------------------------------+-------------------+----------+
|-2SSg0C_LmdI_e1sf0|LIONSGATEPLAY_MOVIE_THETAJSTORYY2025MHI                        |9582.877           |2026-03-21|
|-2SSg0C_LmdI_e1sf0|ZEEFIVE_MOVIE_0-0-1z5133458                                    |7664.0             |2026-03-13|
|-2SSg0C_LmdI_e1sf0|MINITV_TVSHOW_amzn1.dv.gti.9de167a6-7185-4687-a3ca-8ac4198143d2|1027.897           |2026-03-21|
|-2SSg0C_LmdI_e1sf0|SONYLIV_VOD_TVSHOW_1700000196                                  |13200.022          |2026-03-17|
|-2SSg0C_LmdI_e1sf0|ZEEFIVE_MOVIE_0-0-73415                                        |14.0               |2026-03-13|
+------------------+----------------------------------------------------

Sample of indexed train data:


...

26/03/23 05:32:42 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 05:32:42 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 05:32:42 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 05:32:49 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 05:

Train_pdf sample:
Fetching items metadata from Spark to Pandas...


26/03/23 05:33:01 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.


Items_pdf sample uncleaned:
Cleaning missing values and building features...
Filtering orphan items from metadata...
 -> Reduced items metadata to 10776 valid items.
Sample of cleaned items metadata:
Extracted 80 unique item features.
Sample of extracted item features:
['', 'ko', 'hi', 'Sci-Fi & Fantasy', 'pa', 'ml', 'bg', 'ki', 'Reality', 'Thriller']
Fitting LightFM Dataset mapping...
Building Interactions matrices...
Building Item Features matrix...
cell executed successfully!


26/03/23 05:34:05 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 3 for reason Executor for container container_1764236692086_5520_01_000007 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
26/03/23 05:34:35 WARN YarnSchedulerBackend$YarnSchedulerEndpoint: Requesting driver to remove executor 59 for reason Executor for container container_1764236692086_5520_01_000087 exited because of a YARN event (e.g., preemption) and not because of an error in the running job.
